In [1]:
import kagglehub
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import json
from pathlib import Path
import pandas as pd


def load_dance_dataframe(dataset_path='data/dance_dataset.jsonl'):
    """Load dance data from JSONL (preferred) or CSV and normalize key columns."""
    candidate_paths = [
        Path(dataset_path),
        Path('../data/dance_dataset.jsonl'),
        Path('/Users/hannes/Documents/GitHub/KG_Personal_Project/data/dance_dataset.jsonl')
    ]

    source_path = None
    for candidate in candidate_paths:
        if candidate.exists():
            source_path = candidate
            break

    if source_path is None:
        raise FileNotFoundError(
            'Could not find dance dataset. Tried: ' + ', '.join(str(p) for p in candidate_paths)
        )

    if source_path.suffix.lower() == '.jsonl':
        records = []
        with source_path.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        dataframe = pd.DataFrame(records)
    elif source_path.suffix.lower() == '.csv':
        dataframe = pd.read_csv(source_path, sep=',', encoding='latin1')
    else:
        raise ValueError(f'Unsupported file format: {source_path.suffix}')

    # Backward-compatible normalization for old CSV headers.
    if 'dance_type' not in dataframe.columns and 'Dance Type' in dataframe.columns:
        dataframe['dance_type'] = dataframe['Dance Type']
    if 'dance_style' not in dataframe.columns and 'Dance style' in dataframe.columns:
        dataframe['dance_style'] = dataframe['Dance style']

    if 'dance_type' not in dataframe.columns or 'dance_style' not in dataframe.columns:
        raise ValueError("Dataset must contain 'dance_type' and 'dance_style' fields")

    return dataframe


dat_Dancing = load_dance_dataframe()
# Load YouTube API key
def load_youtube_api_key():
    """Load the YouTube API key from yt_key.txt file"""
    with open('yt_key.txt', 'r') as f:
        line = f.read().strip()
        if line.startswith('key='):
            return line.split('=', 1)[1]
        return line

# Initialize YouTube API client
def get_youtube_client():
    """Create and return a YouTube API client"""
    api_key = load_youtube_api_key()
    return build('youtube', 'v3', developerKey=api_key)

def search_videos(youtube, query, max_results=5):
    """
    Search for videos on YouTube

    Args:
        youtube: YouTube API client
        query: Search query string
        max_results: Maximum number of results to return (default: 5)

    Returns:
        List of video information dictionaries
    """
    try:
        request = youtube.search().list(
            part='snippet',
            q=query,
            type='video',
            videoDuration='medium',  # Excludes shorts (short=<4min, medium=4-20min, long=>20min)
            maxResults=max_results
        )
        response = request.execute()

        videos = []
        for item in response.get('items', []):
            video_info = {
                'video_id': item['id']['videoId'],
                'title': item['snippet']['title'],
                'description': item['snippet']['description'],
                'channel': item['snippet']['channelTitle'],
                'published_at': item['snippet']['publishedAt']
            }
            videos.append(video_info)

        return videos

    except HttpError as e:
        print(f"An HTTP error occurred: {e}")
        return []

# Example: Get video details
def get_video_details(youtube, video_id):
    """
    Get detailed information about a specific video

    Args:
        youtube: YouTube API client
        video_id: YouTube video ID

    Returns:
        Dictionary with video details
    """
    try:
        request = youtube.videos().list(
            part='snippet,statistics,contentDetails',
            id=video_id
        )
        response = request.execute()

        if response['items']:
            item = response['items'][0]
            return {
                'title': item['snippet']['title'],
                'description': item['snippet']['description'],
                'channel': item['snippet']['channelTitle'],
                'published_at': item['snippet']['publishedAt'],
                'view_count': item['statistics'].get('viewCount', 0),
                'like_count': item['statistics'].get('likeCount', 0),
                'comment_count': item['statistics'].get('commentCount', 0),
                'duration': item['contentDetails']['duration']
            }
        return None

    except HttpError as e:
        print(f"An HTTP error occurred: {e}")
        return None

# Example: Get video comments
def get_video_comments(youtube, video_id, max_results=20):
    """
    Get comments for a specific video

    Args:
        youtube: YouTube API client
        video_id: YouTube video ID
        max_results: Maximum number of comments to return (default: 20)

    Returns:
        List of comment dictionaries
    """
    try:
        request = youtube.commentThreads().list(
            part='snippet',
            videoId=video_id,
            maxResults=max_results,
            textFormat='plainText',
            order='relevance'  # Can also be 'time' for chronological
        )
        response = request.execute()

        comments = []
        for item in response.get('items', []):
            comment = item['snippet']['topLevelComment']['snippet']
            comment_info = {
                'author': comment['authorDisplayName'],
                'text': comment['textDisplay'],
                'like_count': comment['likeCount'],
                'published_at': comment['publishedAt'],
                'reply_count': item['snippet']['totalReplyCount']
            }
            comments.append(comment_info)

        return comments

    except HttpError as e:
        print(f"An HTTP error occurred: {e}")
        if 'commentsDisabled' in str(e):
            print("Comments are disabled for this video.")
        return []

# Example: Get comprehensive video information
def get_comprehensive_video_info(youtube, video_id):
    """
    Get comprehensive information about a video including all available details

    Args:
        youtube: YouTube API client
        video_id: YouTube video ID

    Returns:
        Dictionary with comprehensive video information
    """
    try:
        request = youtube.videos().list(
            part='snippet,statistics,contentDetails,topicDetails,status',
            id=video_id
        )
        response = request.execute()

        if response['items']:
            item = response['items'][0]
            snippet = item['snippet']
            statistics = item['statistics']
            content_details = item['contentDetails']

            info = {
                # Basic info
                'video_id': video_id,
                'title': snippet['title'],
                'description': snippet['description'],
                'channel_id': snippet['channelId'],
                'channel_title': snippet['channelTitle'],
                'published_at': snippet['publishedAt'],

                # Tags and categories
                'tags': snippet.get('tags', []),
                'category_id': snippet.get('categoryId'),

                # Statistics
                'view_count': statistics.get('viewCount', 0),
                'like_count': statistics.get('likeCount', 0),
                'comment_count': statistics.get('commentCount', 0),

                # Content details
                'duration': content_details['duration'],
                'dimension': content_details.get('dimension'),  # 2d or 3d
                'definition': content_details.get('definition'),  # hd or sd
                'caption': content_details.get('caption'),  # true/false if captions available

                # Topic details (if available)
                'topic_categories': item.get('topicDetails', {}).get('topicCategories', []),

                # Status
                'privacy_status': item.get('status', {}).get('privacyStatus'),
                'license': item.get('status', {}).get('license'),
            }

            return info
        return None

    except HttpError as e:
        print(f"An HTTP error occurred: {e}")
        return None

# Example: Get channel information
def get_channel_info(youtube, channel_id):
    """
    Get information about a YouTube channel

    Args:
        youtube: YouTube API client
        channel_id: YouTube channel ID

    Returns:
        Dictionary with channel information
    """
    try:
        request = youtube.channels().list(
            part='snippet,statistics,contentDetails',
            id=channel_id
        )
        response = request.execute()

        if response['items']:
            item = response['items'][0]
            return {
                'channel_id': channel_id,
                'title': item['snippet']['title'],
                'description': item['snippet']['description'],
                'custom_url': item['snippet'].get('customUrl'),
                'published_at': item['snippet']['publishedAt'],
                'subscriber_count': item['statistics'].get('subscriberCount', 0),
                'video_count': item['statistics'].get('videoCount', 0),
                'view_count': item['statistics'].get('viewCount', 0),
            }
        return None

    except HttpError as e:
        print(f"An HTTP error occurred: {e}")
        return None


/opt/anaconda3/envs/lab3_py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/lab3_py39/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/opt/anaconda3/envs/lab3_py39/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then up

In [2]:
# Function to scrape YouTube data for all dances in the dataset
def scrape_dance_videos(youtube, dataframe, max_videos=2, max_comments=10):
    """
    Scrape YouTube videos and comments for each dance style in the dataset

    Args:
        youtube: YouTube API client
        dataframe: Pandas dataframe with dance data (must have 'dance_type' and 'dance_style' columns)
        max_videos: Number of videos to fetch per dance (default: 2)
        max_comments: Number of comments to fetch per video (default: 10)

    Returns:
        Dictionary with all scraped data organized by dance style
    """
    all_dance_data = {}
    total_dances = len(dataframe)

    print(f"Starting to scrape YouTube data for {total_dances} dance styles...")
    print("=" * 80)

    for idx, row in dataframe.iterrows():
        dance_type = row.get('dance_type', row.get('Dance Type', ''))
        dance_style = row.get('dance_style', row.get('Dance style', ''))

        # Skip if missing data
        if pd.isna(dance_type) or pd.isna(dance_style) or not dance_style:
            print(f"Skipping row {idx}: Missing dance type or style")
            continue

        # Create search query: "{Dance Type} {Dance Style} tutorial"
        query = f"{dance_type} {dance_style} tutorial"

        print(f"\n[{idx+1}/{total_dances}] Processing: {dance_style} ({dance_type})")
        print(f"Query: '{query}'")

        # Search for videos
        videos = search_videos(youtube, query, max_results=max_videos)

        dance_key = f"{dance_type}__{dance_style}"

        if not videos:
            print(f"  ⚠️  No videos found for '{query}'")
            all_dance_data[dance_key] = {
                'dance_type': dance_type,
                'dance_style': dance_style,
                'query': query,
                'videos': []
            }
            continue

        print(f"  ✓ Found {len(videos)} video(s)")

        # Get detailed info and comments for each video
        video_data_list = []
        for video_idx, video in enumerate(videos, 1):
            video_id = video['video_id']
            print(f"    Video {video_idx}: {video['title'][:60]}...")

            # Get comprehensive video info
            comprehensive_info = get_comprehensive_video_info(youtube, video_id)

            # Get comments
            raw_comments = get_video_comments(youtube, video_id, max_results=max_comments)
            print(f"      -> Retrieved {len(raw_comments)} comment(s)")

            # Restructure comments with video_id and comment_number
            structured_comments = []
            for comment_num, comment in enumerate(raw_comments, 1):
                structured_comment = {
                    'video_id': video_id,
                    'comment_number': comment_num,
                    'text': comment['text'],
                    'like_count': comment['like_count'],
                    'published_at': comment['published_at'],
                    'reply_count': comment['reply_count']
                }
                structured_comments.append(structured_comment)

            # Combine all video data
            video_data = {
                'basic_info': video,
                'comprehensive_info': comprehensive_info,
                'comments': structured_comments,
                'url': f"https://www.youtube.com/watch?v={video_id}"
            }
            video_data_list.append(video_data)

        # Store data for this dance style
        all_dance_data[dance_key] = {
            'dance_type': dance_type,
            'dance_style': dance_style,
            'query': query,
            'videos': video_data_list
        }

    print("\n" + "=" * 80)
    print(f"✓ Scraping complete! Collected data for {len(all_dance_data)} dance styles")

    return all_dance_data


In [3]:
# Execute the scraping
print("Initializing YouTube API client...")
youtube = get_youtube_client()
print("✓ YouTube client initialized\n")

# Scrape data for all dances (top 2 videos with 10 comments each)
dance_youtube_data = scrape_dance_videos(youtube, dat_Dancing, max_videos=2, max_comments=10)

# Save the results to a JSON file for later use
output_file = '../data/yt_data/dance_youtube_data.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(dance_youtube_data, f, indent=2, ensure_ascii=False)

print(f"\n✓ Data saved to {output_file}")


Initializing YouTube API client...
✓ YouTube client initialized

Starting to scrape YouTube data for 130 dance styles...

[1/130] Processing: Tap dancing (American)
Query: 'American Tap dancing tutorial'
  ✓ Found 2 video(s)
    Video 1: How to TAP DANCE - Beginner Tutorial...
      -> Retrieved 10 comment(s)
    Video 2: TAP DANCE BASICS - 5 Steps EVERY Beginner should Master...
      -> Retrieved 10 comment(s)

[2/130] Processing: Stepping (American)
Query: 'American Stepping tutorial'
  ✓ Found 2 video(s)
    Video 1: Let Us Show You How .... Chicago Style Stepping ... Effortle...
      -> Retrieved 10 comment(s)
    Video 2: How to TAP DANCE - Beginner Tutorial...
      -> Retrieved 10 comment(s)

[3/130] Processing: Jazz dance (American)
Query: 'American Jazz dance tutorial'
  ✓ Found 2 video(s)
    Video 1: 10 Basic Jazz Dance Moves...
      -> Retrieved 10 comment(s)
    Video 2: Solo Jazz warm-up dance to Honeysuckle Rose...
      -> Retrieved 10 comment(s)

[4/130] Processing:

In [4]:
# Display summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

total_dances = len(dance_youtube_data)
total_videos = sum(len(data['videos']) for data in dance_youtube_data.values())
total_comments = sum(
    len(video['comments'])
    for data in dance_youtube_data.values()
    for video in data['videos']
)

print(f"Total dance styles processed: {total_dances}")
print(f"Total videos collected: {total_videos}")
print(f"Total comments collected: {total_comments}")
print(f"Average videos per dance: {total_videos/total_dances:.2f}")
print(f"Average comments per video: {total_comments/total_videos:.2f}" if total_videos > 0 else "N/A")



SUMMARY STATISTICS
Total dance styles processed: 130
Total videos collected: 527
Total comments collected: 3760
Average videos per dance: 4.05
Average comments per video: 7.13


In [5]:
# Display a sample of the collected data
print("\n" + "=" * 80)
print("SAMPLE DATA (First Dance Style)")
print("=" * 80)

if dance_youtube_data:
    first_dance = list(dance_youtube_data.keys())[0]
    sample_data = dance_youtube_data[first_dance]

    print(f"\nDance Style: {sample_data.get('dance_style', first_dance)}")
    print(f"Dance Type: {sample_data['dance_type']}")
    print(f"Query Used: {sample_data['query']}")
    print(f"Number of Videos: {len(sample_data['videos'])}")

    for idx, video in enumerate(sample_data['videos'], 1):
        print(f"\n  Video {idx}:")
        print(f"    Title: {video['basic_info']['title']}")
        print(f"    URL: {video['url']}")
        print(f"    Channel: {video['basic_info']['channel']}")
        print(f"    Published: {video['basic_info']['published_at']}")

        if video['comprehensive_info']:
            print(f"    Views: {int(video['comprehensive_info']['view_count']):,}")
            print(f"    Likes: {int(video['comprehensive_info']['like_count']):,}")
            print(f"    Duration: {video['comprehensive_info']['duration']}")

        print(f"    Comments ({len(video['comments'])}):")
        for comment in video['comments'][:3]:
            comment_text = comment['text'][:80].replace('\n', ' ')
            print(f"      #{comment['comment_number']} (video: {comment['video_id'][:8]}...): {comment_text}...")



SAMPLE DATA (First Dance Style)

Dance Style: Tap dancing
Dance Type: American
Query Used: American Tap dancing tutorial
Number of Videos: 2

  Video 1:
    Title: How to TAP DANCE - Beginner Tutorial
    URL: https://www.youtube.com/watch?v=LH_EyxsupRE
    Channel: Taptopia
    Published: 2016-04-19T10:52:47Z
    Views: 2,978,419
    Likes: 52,436
    Duration: PT8M18S
    Comments (10):
      #1 (video: LH_Eyxsu...): Learning tap was my secret childhood dream. I just ordered a pair of tap shoes (...
      #2 (video: LH_Eyxsu...): Me at 3PM: I should really spend my night studying today, then get enough sleep....
      #3 (video: LH_Eyxsu...): Wow... i took a class and i learned more in this 8 mins than in the 45 min class...

  Video 2:
    Title: TAP DANCE BASICS - 5 Steps EVERY Beginner should Master
    URL: https://www.youtube.com/watch?v=HZruz8wO0g8
    Channel: The Dance Prof
    Published: 2018-03-11T00:00:01Z
    Views: 425,730
    Likes: 11,928
    Duration: PT9M31S
    Com

In [6]:
# Create a pandas DataFrame for easier analysis
def create_videos_dataframe(dance_youtube_data):
    """
    Convert the scraped YouTube data into a flat pandas DataFrame
    """
    rows = []

    for _, data in dance_youtube_data.items():
        dance_style = data.get('dance_style', '')
        dance_type = data['dance_type']
        query = data['query']

        for video in data['videos']:
            basic = video['basic_info']
            comp = video['comprehensive_info']

            row = {
                'dance_style': dance_style,
                'dance_type': dance_type,
                'search_query': query,
                'video_id': basic['video_id'],
                'video_url': video['url'],
                'title': basic['title'],
                'description': basic['description'],
                'channel': basic['channel'],
                'published_at': basic['published_at'],
                'view_count': int(comp['view_count']) if comp else 0,
                'like_count': int(comp['like_count']) if comp else 0,
                'comment_count': int(comp['comment_count']) if comp else 0,
                'duration': comp['duration'] if comp else None,
                'num_comments_retrieved': len(video['comments'])
            }
            rows.append(row)

    return pd.DataFrame(rows)

# Create DataFrame
videos_df = create_videos_dataframe(dance_youtube_data)

# Display the DataFrame
print("\n" + "=" * 80)
print("VIDEOS DATAFRAME")
print("=" * 80)
print(videos_df.head(10))

# Save to CSV
videos_csv = 'dance_videos.csv'
videos_df.to_csv(videos_csv, index=False, encoding='utf-8')
print(f"\n✓ Videos data saved to {videos_csv}")



VIDEOS DATAFRAME
                        dance_style   dance_type  \
0                       Tap dancing     American   
1                       Tap dancing     American   
2                          Stepping     American   
3                          Stepping     American   
4                        Jazz dance     American   
5                        Jazz dance     American   
6  Moonwalk (Michael Jackson style)     American   
7  Moonwalk (Michael Jackson style)     American   
8                       Raqs Sharqi  Belly dance   
9                       Raqs Sharqi  Belly dance   

                                        search_query     video_id  \
0                      American Tap dancing tutorial  LH_EyxsupRE   
1                      American Tap dancing tutorial  HZruz8wO0g8   
2                         American Stepping tutorial  OSa4tPmduk8   
3                         American Stepping tutorial  LH_EyxsupRE   
4                       American Jazz dance tutorial  dX7Tz_VBfK

In [7]:
# Show some interesting statistics
print("\n" + "=" * 80)
print("INTERESTING STATISTICS")
print("=" * 80)

print("\nTop 5 Most Viewed Videos:")
print(videos_df.nlargest(5, 'view_count')[['dance_style', 'title', 'view_count', 'video_url']])

print("\nTop 5 Most Liked Videos:")
print(videos_df.nlargest(5, 'like_count')[['dance_style', 'title', 'like_count', 'video_url']])

print("\nDance Styles with Most Comments:")
comment_stats = videos_df.groupby('dance_style')['comment_count'].sum().sort_values(ascending=False).head(10)
print(comment_stats)



INTERESTING STATISTICS

Top 5 Most Viewed Videos:
    dance_style                                              title  \
476        Frug  The Dead South - In Hell I&#39;ll Be In Good C...   
464        Frug  The Dance Freeze Song | Freeze Dance | Scratch...   
469        Frug  Chappell Roan - Pink Pony Club (Official Music...   
463        Frug  Brain Breaks - Action Songs for Children - Mov...   
468        Frug    Feel Good Inc. (metal cover by Leo Moracchioli)   

     view_count                                    video_url  
476   466541088  https://www.youtube.com/watch?v=B9FzVhw8_bY  
464   386088922  https://www.youtube.com/watch?v=A1vdKfXlB_g  
469   123238365  https://www.youtube.com/watch?v=GR3Liudev18  
463    58256381  https://www.youtube.com/watch?v=388Q44ReOWE  
468    54320160  https://www.youtube.com/watch?v=yNENVZFHutQ  

Top 5 Most Liked Videos:
    dance_style                                              title  \
476        Frug  The Dead South - In Hell I&#39;ll Be 